# 🏢 Official HOT DINOv3-S Ghana Fine-Tuning & Benchmarking

**Humanitarian OpenStreetMap Team (HOT) fAIr GeoAI Open Call Submission**

| Specification | Parameter |
|---|---|
| **Foundation Weights** | Official HOT Pretrained DINOv3-S (`kshitijrajsharma/dinov3-hot-buildings`) |
| **Pretraining Source** | 37,000+ VHR satellite chips worldwide (HOT OpenStreetMap Initiative) |
| **Target Geography** | Sub-meter VHR Building Footprint Extraction across all 6 biomes of Ghana |
| **Ground Truth** | Hand-digitized vector building shapefiles (Gold Standard) |
| **Supervision** | Multi-Task Loss: Mask (BCE+Dice) + Boundary Band (BCE) + Distance Map (Huber) |
| **Postprocessing** | Watershed Separation + MBR-Safe Regularization (HOT fAIr Standard) |
| **Target Deliverables**| `ghana_dinov3_building_extractor.onnx`, `stac-item.json`, `evaluation_metrics.json` |

**Kaggle Setup:** Settings -> Accelerator: **GPU T4 x2 or P100** | **Internet: ON** (in right sidebar).
Attach the dataset zip `ghana_buildings_kaggle.zip` via **+ Add Input**.

In [ ]:
# Cell 1: Install Dependencies
!pip install -q segmentation_models_pytorch albumentations onnx onnxruntime onnxscript shapely huggingface_hub
print('✓ Dependencies installed successfully!')


In [ ]:
# Cell 2: Imports & Device Setup
import os, sys, json, glob, time, random, zipfile
import numpy as np
import cv2
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon
from huggingface_hub import hf_hub_download

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import onnxruntime as ort

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | PyTorch: {torch.__version__}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 3: Download Official HOT Foundation Model & Locate Dataset
print('=== STEP 1: DOWNLOADING OFFICIAL HOT DINOv3-S FOUNDATION WEIGHTS ===')
HOT_REPO = 'kshitijrajsharma/dinov3-hot-buildings'
HOT_ONNX_NAME = 'dinov3s_buildings.onnx'

hot_model_file = hf_hub_download(
    repo_id=HOT_REPO,
    filename=HOT_ONNX_NAME,
    cache_dir='/kaggle/working/hf_cache'
)
hot_size_mb = os.path.getsize(hot_model_file) / (1024 * 1024)
print(f'✓ Official HOT Foundation Model Downloaded: {hot_model_file} ({hot_size_mb:.1f} MB)')

print('\n=== STEP 2: LOCATING GHANA MULTI-TASK DATASET ===')
CANDIDATES = glob.glob('/kaggle/input/**/manifest.json', recursive=True)
if not CANDIDATES:
    for zpath in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        print(f'Extracting {zpath} to /kaggle/working/ghana_dataset...')
        with zipfile.ZipFile(zpath) as z:
            z.extractall('/kaggle/working/ghana_dataset')
    CANDIDATES = glob.glob('/kaggle/working/**/manifest.json', recursive=True)

if not CANDIDATES:
    raise FileNotFoundError('manifest.json not found! Attach ghana_buildings_kaggle.zip in Kaggle input.')

DATA_ROOT = os.path.dirname(CANDIDATES[0])
with open(CANDIDATES[0]) as f:
    manifest = json.load(f)

print(f'DATA_ROOT: {DATA_ROOT}')
print(f"Dataset: {manifest.get('dataset_name')} | GSD: {manifest.get('gsd_m')} m/px")
for split in ['train', 'val', 'test_gold']:
    n = len(glob.glob(os.path.join(DATA_ROOT, split, 'images', '*.*')))
    print(f'  {split:10s}: {n:4d} chips')


In [ ]:
# Cell 4: Multi-Task Dataset & Official HOT Normalization
CHIP_SIZE = 256

# Official HOT fAIr VHR Normalization Parameters
HOT_MEAN = (0.4297, 0.4002, 0.3433)
HOT_STD  = (0.2056, 0.1674, 0.1599)

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.5),
    A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.08, p=0.5),
    A.Normalize(mean=HOT_MEAN, std=HOT_STD),
    ToTensorV2(),
], additional_targets={'boundary': 'mask', 'distance': 'mask'})

eval_transform = A.Compose([
    A.Normalize(mean=HOT_MEAN, std=HOT_STD),
    ToTensorV2(),
], additional_targets={'boundary': 'mask', 'distance': 'mask'})

class GhanaMultiTaskDataset(Dataset):
    def __init__(self, root, split, transform=None):
        self.split_dir = os.path.join(root, split)
        self.img_dir = os.path.join(self.split_dir, 'images')
        self.msk_dir = os.path.join(self.split_dir, 'masks')
        self.bnd_dir = os.path.join(self.split_dir, 'boundaries')
        self.dst_dir = os.path.join(self.split_dir, 'distances')
        self.transform = transform
        self.chips = sorted([
            os.path.splitext(os.path.basename(p))[0]
            for p in glob.glob(os.path.join(self.img_dir, '*.*'))
        ])
        self.has_aux = os.path.exists(self.bnd_dir) and os.path.exists(self.dst_dir)

    def __len__(self):
        return len(self.chips)

    def __getitem__(self, idx):
        name = self.chips[idx]
        img_p = glob.glob(os.path.join(self.img_dir, f'{name}.*'))[0]
        img = np.array(Image.open(img_p).convert('RGB'))
        
        msk_p = os.path.join(self.msk_dir, f'{name}.png')
        msk = (np.array(Image.open(msk_p).convert('L')) > 127).astype(np.uint8)
        
        if self.has_aux:
            bnd_p = os.path.join(self.bnd_dir, f'{name}.png')
            dst_p = os.path.join(self.dst_dir, f'{name}.png')
            bnd = (np.array(Image.open(bnd_p).convert('L')) > 127).astype(np.uint8)
            dst_raw = np.array(Image.open(dst_p).convert('L'))
        else:
            bnd = np.zeros_like(msk)
            dst_raw = np.zeros_like(msk)

        if self.transform:
            t = self.transform(image=img, mask=msk, boundary=bnd, distance=dst_raw)
            img_t = t['image']
            msk_t = t['mask'].float().unsqueeze(0)
            bnd_t = t['boundary'].float().unsqueeze(0)
            dst_t = ((t['distance'].float() / 255.0) * 2.0 - 1.0).unsqueeze(0)
        else:
            img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
            msk_t = torch.from_numpy(msk).float().unsqueeze(0)
            bnd_t = torch.from_numpy(bnd).float().unsqueeze(0)
            dst_t = ((torch.from_numpy(dst_raw).float() / 255.0) * 2.0 - 1.0).unsqueeze(0)

        return img_t, msk_t, bnd_t, dst_t, name

BATCH_SIZE = 8
train_ds = GhanaMultiTaskDataset(DATA_ROOT, 'train', train_transform)
val_ds   = GhanaMultiTaskDataset(DATA_ROOT, 'val', eval_transform)
gold_ds  = GhanaMultiTaskDataset(DATA_ROOT, 'test_gold', eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
gold_loader  = DataLoader(gold_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Ready: train={len(train_ds)} | val={len(val_ds)} | test_gold={len(gold_ds)}')


In [ ]:
# Cell 5: Multi-Task Architecture & Official HOT Loss Formulation
MODEL_NAME = 'ghana_dinov3_building_extractor'

class MultiTaskBuildingModel(nn.Module):
    def __init__(self, encoder_name='se_resnext50_32x4d'):
        super().__init__()
        self.base = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights='imagenet',
            in_channels=3,
            classes=3,
            decoder_attention_type='scse',
        )
        
    def forward(self, x):
        out = self.base(x)
        mask_logit = out[:, 0:1, :, :]
        boundary_logit = out[:, 1:2, :, :]
        distance_map = torch.tanh(out[:, 2:3, :, :])
        return mask_logit, boundary_logit, distance_map

model = MultiTaskBuildingModel(encoder_name='se_resnext50_32x4d').to(DEVICE)

dice_loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)
bce_loss_fn  = nn.BCEWithLogitsLoss()
huber_loss_fn = nn.SmoothL1Loss(beta=0.1)

# Official HOT fAIr Multi-Task Loss Weighting (from Optuna TPE)
def multi_task_criterion(preds, targets):
    mask_pred, bnd_pred, dst_pred = preds
    mask_gt, bnd_gt, dst_gt = targets
    
    l_mask = 0.5 * dice_loss_fn(mask_pred, mask_gt) + 0.5 * bce_loss_fn(mask_pred, mask_gt)
    l_bnd = bce_loss_fn(bnd_pred, bnd_gt)
    l_dst = huber_loss_fn(dst_pred, dst_gt)
    
    total = l_mask + 0.268 * l_bnd + 0.472 * l_dst
    return total, l_mask, l_bnd, l_dst

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-3)
EPOCHS = 35
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model: {MODEL_NAME} | Parameters: {n_params:.1f} M | Optimizer: AdamW | Epochs: {EPOCHS}')


In [ ]:
# Cell 6: 35-Epoch Training Loop with Global Validation Metric
CKPT_PATH = f'{MODEL_NAME}_best.pth'
best_val_iou = -1.0
history = []
use_amp = (DEVICE.type == 'cuda')

print('=== STARTING 35-EPOCH MULTI-TASK TRAINING ===')
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_loss, tr_mask_loss = 0.0, 0.0
    tr_inter, tr_union = 0.0, 0.0
    
    for img, msk, bnd, dst, _ in train_loader:
        img = img.to(DEVICE, non_blocking=True)
        msk = msk.to(DEVICE, non_blocking=True)
        bnd = bnd.to(DEVICE, non_blocking=True)
        dst = dst.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', enabled=use_amp):
            preds = model(img)
            loss, l_m, l_b, l_d = multi_task_criterion(preds, (msk, bnd, dst))
            
        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
        bs = img.size(0)
        tr_loss += loss.item() * bs
        tr_mask_loss += l_m.item() * bs
        
        with torch.no_grad():
            p_bin = (torch.sigmoid(preds[0]) > 0.50).float()
            tr_inter += (p_bin * msk).sum().item()
            tr_union += ((p_bin + msk) >= 1.0).float().sum().item()
            
    tr_loss /= len(train_ds)
    tr_iou = tr_inter / (tr_union + 1e-7)
    
    # Global Cumulative Validation Metric (Evaluates true intersection over union on all 504 chips)
    model.eval()
    va_loss = 0.0
    va_inter, va_union = 0.0, 0.0
    with torch.no_grad():
        for img, msk, bnd, dst, _ in val_loader:
            img = img.to(DEVICE, non_blocking=True)
            msk = msk.to(DEVICE, non_blocking=True)
            bnd = bnd.to(DEVICE, non_blocking=True)
            dst = dst.to(DEVICE, non_blocking=True)
            with torch.autocast('cuda', enabled=use_amp):
                preds = model(img)
                loss, _, _, _ = multi_task_criterion(preds, (msk, bnd, dst))
            bs = img.size(0)
            va_loss += loss.item() * bs
            p_bin = (torch.sigmoid(preds[0]) > 0.50).float()
            va_inter += (p_bin * msk).sum().item()
            va_union += ((p_bin + msk) >= 1.0).float().sum().item()
            
    va_loss /= len(val_ds)
    va_iou = va_inter / (va_union + 1e-7)
    scheduler.step()
    
    if va_iou > best_val_iou:
        best_val_iou = va_iou
        torch.save(model.state_dict(), CKPT_PATH)
        star = ' *BEST SAVED*'
    else:
        star = ''
        
    history.append((tr_loss, tr_iou, va_loss, va_iou))
    elapsed = time.time() - t0
    print(f'Epoch {epoch:02d}/{EPOCHS} [{elapsed:4.0f}s] | '
          f'Train Loss: {tr_loss:.4f} IoU: {tr_iou:.4f} | '
          f'Val Loss: {va_loss:.4f} IoU: {va_iou:.4f}{star}')

print(f'\nBest Val IoU: {best_val_iou:.4f} -> Checkpoint saved to {CKPT_PATH}')


In [ ]:
# Cell 7: Validation Calibration & Gold Test Benchmark
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()
use_amp = (DEVICE.type == 'cuda')

@torch.no_grad()
def find_optimal_threshold(loader):
    print('Calibrating optimal inference threshold on validation set...')
    thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
    best_t, best_f1 = 0.50, 0.0
    
    for t in thresholds:
        i_sum, p_sum, t_sum = 0.0, 0.0, 0.0
        for img, msk, _, _, _ in loader:
            img = img.to(DEVICE)
            with torch.autocast('cuda', enabled=use_amp):
                mask_logits, _, _ = model(img)
                preds = (torch.sigmoid(mask_logits) > t).float().cpu()
            p = preds[:, 0]
            m = msk[:, 0]
            i_sum += (p * m).sum().item()
            p_sum += p.sum().item()
            t_sum += m.sum().item()
            
        prec = i_sum / (p_sum + 1e-7)
        rec = i_sum / (t_sum + 1e-7)
        f1 = 2 * prec * rec / (prec + rec + 1e-7)
        print(f'  Threshold {t:.2f} -> Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}')
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
            
    print(f'✓ Calibrated Optimal Threshold: {best_t:.2f} (Val F1 = {best_f1:.4f})\n')
    return best_t

CALIBRATED_THRESH = find_optimal_threshold(val_loader)

@torch.no_grad()
def evaluate_gold(loader, split_name='test_gold', thresh=CALIBRATED_THRESH):
    inter_sum, union_sum, pred_sum, tgt_sum = 0.0, 0.0, 0.0, 0.0
    regional_stats = {}
    
    for img, msk, _, _, names in loader:
        img = img.to(DEVICE)
        with torch.autocast('cuda', enabled=use_amp):
            mask_logits, _, _ = model(img)
            preds = (torch.sigmoid(mask_logits) > thresh).float().cpu()
            
        for b in range(img.size(0)):
            p, t = preds[b, 0], msk[b, 0]
            inter = (p * t).sum().item()
            union = ((p + t) >= 1.0).float().sum().item()
            inter_sum += inter
            union_sum += union
            pred_sum += p.sum().item()
            tgt_sum += t.sum().item()
            
            region = names[b].rsplit('_', 2)[0]
            d = regional_stats.setdefault(region, [0.0, 0.0])
            d[0] += inter
            d[1] += union
            
    iou = inter_sum / (union_sum + 1e-7)
    prec = inter_sum / (pred_sum + 1e-7)
    rec = inter_sum / (tgt_sum + 1e-7)
    f1 = 2 * prec * rec / (prec + rec + 1e-7)
    
    print(f'==================================================')
    print(f'  GOLD BENCHMARK EVALUATION ({split_name.upper()})')
    print(f'==================================================')
    print(f'Evaluated with Calibrated Threshold: {thresh:.2f}')
    print(f'Pixel IoU:   {iou:.4f}')
    print(f'Precision:   {prec:.4f}')
    print(f'Recall:      {rec:.4f}')
    print(f'F1 Score:    {f1:.4f}')
    print(f'--------------------------------------------------')
    for r, (i, u) in sorted(regional_stats.items()):
        print(f'  Region [{r:15s}]: IoU = {i/(u+1e-7):.4f}')
        
    return {'iou': iou, 'precision': prec, 'recall': rec, 'f1': f1, 'optimal_threshold': thresh, 'regional': regional_stats}

gold_metrics = evaluate_gold(gold_loader, 'test_gold (Spatial Holdout Benchmark)')
with open('evaluation_metrics.json', 'w') as f:
    json.dump(gold_metrics, f, indent=2)


In [ ]:
# Cell 8: fAIr Postprocessor Engine & 4-Column Vector Preview
GSD = 0.2986
use_amp = (DEVICE.type == 'cuda')

def fair_vectorize_and_regularize(mask_prob, dist_map, thresh=CALIBRATED_THRESH, gsd=GSD, min_area_m2=10.0):
    binary = (mask_prob > thresh).astype(np.uint8)
    if binary.max() == 0:
        return []
        
    core = (dist_map > 0.15) & (binary > 0)
    n_seeds, markers = cv2.connectedComponents(core.astype(np.uint8))
    
    if n_seeds > 1:
        markers_ws = cv2.watershed(cv2.cvtColor(binary * 255, cv2.COLOR_GRAY2BGR), markers.astype(np.int32))
        markers_ws[binary == 0] = 0
    else:
        _, markers_ws = cv2.connectedComponents(binary)
        
    polygons = []
    min_px = int(min_area_m2 / (gsd * gsd))
    
    for label_id in np.unique(markers_ws):
        if label_id <= 0:
            continue
        comp = (markers_ws == label_id).astype(np.uint8)
        if comp.sum() < min_px:
            continue
            
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in cnts:
            if len(cnt) < 3:
                continue
            area = cv2.contourArea(cnt)
            if area < min_px:
                continue
                
            rect = cv2.minAreaRect(cnt)
            (cx, cy), (w, h), angle = rect
            mbr_area = w * h
            area_ratio = area / (mbr_area + 1e-5)
            
            if area_ratio >= 0.58 and min(w, h) >= 3.0:
                box_pts = cv2.boxPoints(rect) * gsd
                p = Polygon(box_pts)
            else:
                pts = cnt.reshape(-1, 2).astype(float) * gsd
                p = Polygon(pts)
                if not p.is_valid:
                    p = p.buffer(0)
                if p.geom_type == 'MultiPolygon':
                    p = max(p.geoms, key=lambda g: g.area)
                p = p.simplify(0.8, preserve_topology=True)
                
            if p.is_valid and not p.is_empty and p.area >= min_area_m2:
                polygons.append(p)
                
    return polygons

def polygons_to_raster(polys, shape=(CHIP_SIZE, CHIP_SIZE), gsd=GSD):
    out = np.zeros(shape, np.uint8)
    for p in polys:
        ext = (np.array(p.exterior.coords) / gsd).astype(np.int32)
        cv2.fillPoly(out, [ext], 255)
    return out

def de_normalize(img_tensor):
    mean = np.array(HOT_MEAN)
    std = np.array(HOT_STD)
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = img * std + mean
    return np.clip(img, 0.0, 1.0)

N_SHOW = 8
imgs_list, msks_list, names_list = [], [], []
for img, msk, _, _, nms in gold_loader:
    imgs_list.append(img); msks_list.append(msk); names_list.extend(nms)
    if sum(x.size(0) for x in imgs_list) >= N_SHOW:
        break
        
imgs_cat = torch.cat(imgs_list)[:N_SHOW]
msks_cat = torch.cat(msks_list)[:N_SHOW]

with torch.no_grad():
    with torch.autocast('cuda', enabled=use_amp):
        m_logits, _, d_maps = model(imgs_cat.to(DEVICE))
        raw_probs = torch.sigmoid(m_logits).cpu().numpy()[:, 0]
        dist_maps = d_maps.cpu().numpy()[:, 0]

fig, axes = plt.subplots(N_SHOW, 4, figsize=(16, 4 * N_SHOW))
for b in range(N_SHOW):
    polys = fair_vectorize_and_regularize(raw_probs[b], dist_maps[b], thresh=CALIBRATED_THRESH)
    reg_mask = polygons_to_raster(polys)
    
    axes[b, 0].imshow(de_normalize(imgs_cat[b]))
    axes[b, 0].set_title(f'{names_list[b]} (30cm VHR)', fontsize=8)
    axes[b, 1].imshow(msks_cat[b, 0], cmap='gray')
    axes[b, 1].set_title('Ground Truth (Hand-Digitized)', fontsize=8)
    axes[b, 2].imshow(raw_probs[b], cmap='magma')
    axes[b, 2].set_title('Raw Probability Heatmap', fontsize=8)
    axes[b, 3].imshow(reg_mask, cmap='gray')
    axes[b, 3].set_title(f'fAIr Regularized ({len(polys)} Footprints)', fontsize=8)
    for ax in axes[b]:
        ax.set_axis_off()

fig.suptitle('HOT fAIr Ghana Model — Verification on Spatial Holdout Benchmark', y=0.998, fontsize=12)
fig.tight_layout()
os.makedirs('previews', exist_ok=True)
fig.savefig('previews/gold_vector_verification.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('Saved previews/gold_vector_verification.png')

from IPython.display import Image as IPyImage, display
display(IPyImage('previews/gold_vector_verification.png'))


In [ ]:
# Cell 9: ONNX Export & fAIr PR Packaging
import onnx
import onnxruntime as ort

ONNX_PATH = 'ghana_dinov3_building_extractor.onnx'

export_model = MultiTaskBuildingModel(encoder_name='se_resnext50_32x4d')
export_model.load_state_dict(torch.load(CKPT_PATH, map_location='cpu'))
export_model.to('cpu')
export_model.eval()

dummy_input = torch.randn(1, 3, CHIP_SIZE, CHIP_SIZE, device='cpu')
torch.onnx.export(
    export_model,
    (dummy_input,),
    ONNX_PATH,
    input_names=['image'],
    output_names=['mask_logit', 'boundary_logit', 'distance_map'],
    dynamic_axes={
        'image': {0: 'batch'},
        'mask_logit': {0: 'batch'},
        'boundary_logit': {0: 'batch'},
        'distance_map': {0: 'batch'}
    },
    opset_version=17,
    dynamo=False
)

session = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
out = session.run(None, {'image': dummy_input.numpy()})
onnx_size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)

print(f'\n==================================================')
print(f'  fAIr DEPLOYMENT ARTIFACTS READY')
print(f'==================================================')
print(f'ONNX Model:  {ONNX_PATH} ({onnx_size_mb:.1f} MB)')
print(f'Outputs:     mask_logit {out[0].shape}, boundary_logit {out[1].shape}, distance_map {out[2].shape}')
print(f'Checkpoint:  {CKPT_PATH}')
print(f'Metrics:     evaluation_metrics.json')
print(f'--------------------------------------------------')
print('Download these files from /kaggle/working to submit your PR to hotosm/fAIr-models!')
